In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.3 MB/s eta 0:00:00


Bước 1: đọc file đưa vào dataframe

In [4]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, KFold, StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, RandomForestClassifier



from sklearn.neural_network import MLPClassifier

from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import confusion_matrix
from sklearn.tree import export_graphviz

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer

from catboost import CatBoostClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score

import datetime
import xgboost as xgb


import numpy as np
import time
import datetime
import pydot
import os
import sys
import seaborn as sns
import matplotlib.pyplot as plt



# ĐỌC FILE CNTT_BATBUOC_DAXULY ở bước 1 tiền xử lý --> đưa vào dataframe tên là df
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/HUONG DAN LUAN VAN 2025-2026/DU DOAN UNG THU PHOI/1.1.TienXuLyDuLieu/lung_cancer_preprocessed.csv')

#in dữ liệu sau khi đọc từ file
print(df)

           AGE  GENDER  SMOKING  FINGER_DISCOLORATION  MENTAL_STRESS  \
0     0.682203       1        1                     1              1   
1     1.505110       1        1                     0              0   
2     0.049197       1        1                     0              0   
3    -0.837011       0        1                     0              1   
4     0.935405       0        1                     1              1   
...        ...     ...      ...                   ...            ...   
4995 -1.596618       0        1                     1              0   
4996  1.441810       0        1                     1              1   
4997 -0.393907       1        0                     0              1   
4998  1.188608       1        0                     1              0   
4999 -1.533317       0        1                     0              0   

      EXPOSURE_TO_POLLUTION  LONG_TERM_ILLNESS  ENERGY_LEVEL  IMMUNE_WEAKNESS  \
0                         1                  0      0.

In [5]:
# ===== Đổi tên cột PULMONARY_DISEASE -> target =====
df.rename(columns={'PULMONARY_DISEASE': 'target'}, inplace=True)

print(df)

           AGE  GENDER  SMOKING  FINGER_DISCOLORATION  MENTAL_STRESS  \
0     0.682203       1        1                     1              1   
1     1.505110       1        1                     0              0   
2     0.049197       1        1                     0              0   
3    -0.837011       0        1                     0              1   
4     0.935405       0        1                     1              1   
...        ...     ...      ...                   ...            ...   
4995 -1.596618       0        1                     1              0   
4996  1.441810       0        1                     1              1   
4997 -0.393907       1        0                     0              1   
4998  1.188608       1        0                     1              0   
4999 -1.533317       0        1                     0              0   

      EXPOSURE_TO_POLLUTION  LONG_TERM_ILLNESS  ENERGY_LEVEL  IMMUNE_WEAKNESS  \
0                         1                  0      0.

In [6]:
#kiem tra du lieu các cột còn thiếu giá trị (đếm các cột trống có dữ liệu trống)
df.isna().sum()

,0
AGE,0
GENDER,0
SMOKING,0
FINGER_DISCOLORATION,0
MENTAL_STRESS,0
EXPOSURE_TO_POLLUTION,0
LONG_TERM_ILLNESS,0
ENERGY_LEVEL,0
IMMUNE_WEAKNESS,0
BREATHING_ISSUE,0


In [7]:
#cập nhật lại X, y
X = df.iloc[:, 0: -1]  # lấy tất cả các dòng, còn cột bỏ cột cuối
y = df.iloc[:, -1]  # chỉ lấy cột cuối

#hoặc viết
#X = df.drop(columns=['target'])
#y = df[['target']]


print(X)
print(y)

           AGE  GENDER  SMOKING  FINGER_DISCOLORATION  MENTAL_STRESS  \
0     0.682203       1        1                     1              1   
1     1.505110       1        1                     0              0   
2     0.049197       1        1                     0              0   
3    -0.837011       0        1                     0              1   
4     0.935405       0        1                     1              1   
...        ...     ...      ...                   ...            ...   
4995 -1.596618       0        1                     1              0   
4996  1.441810       0        1                     1              1   
4997 -0.393907       1        0                     0              1   
4998  1.188608       1        0                     1              0   
4999 -1.533317       0        1                     0              0   

      EXPOSURE_TO_POLLUTION  LONG_TERM_ILLNESS  ENERGY_LEVEL  IMMUNE_WEAKNESS  \
0                         1                  0      0.

Tìm các tham số tối ưu cho tất cả các giải thuật Sinh viên chạy nhiều lần để chọn ra tham số nào làm cho Accuracy cao nhất

1.GridSearchCV cho KNN

In [9]:
# Xác định các tham số cho Random Search


param_grid = {
    # k từ 1-20 chi tiết, sau đó 25-100 bước 5
    'n_neighbors': list(range(1, 21)) + list(range(25, 101, 5)),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}


# Tạo mô hình KNN classifier
knn = KNeighborsClassifier()

# Chia dữ liệu thành 10 fold để huấn luyện
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

#random_search = RandomizedSearchCV(knn, param_distributions=param_dist, cv=skf, scoring='accuracy')
#random_search.fit(X, y)



start_time = time.time()

grid_search = GridSearchCV(knn, param_grid, cv=skf, scoring='accuracy')
grid_search.fit(X, y)

end_time = time.time()
elapsed_time = end_time - start_time

# Chuyển giây sang phút + giây
minutes = int(elapsed_time // 60)
seconds = int(elapsed_time % 60)


# Lấy toàn bộ dữ liệu
results = grid_search.cv_results_

# In kết quả
for i, k in enumerate(results['param_n_neighbors']):
    print(f"k = {k}, Accuracy = {results['mean_test_score'][i]}, metric = {results['param_metric'][i]}")

best_knn = grid_search.best_estimator_
print(best_knn)

# In kết quả tốt nhất
best_k = grid_search.best_params_['n_neighbors']
best_score = grid_search.best_score_
best_metric = grid_search.best_params_['metric']
print(f"\nBest value of k: {best_k}")
print(f"Best accuracy score: {best_score}")
print(f"Best metric: {best_metric}")

# Lưu kết quả vào file CSV với ngày giờ chạy
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_file = "/content/drive/MyDrive/Colab Notebooks/HUONG DAN LUAN VAN 2025-2026/DU DOAN UNG THU PHOI/1.2.ChonThamSoToiUu/lungcancer_grid_search_KNN.txt"

with open(output_file, 'w') as txtfile:
    txtfile.write('k, Accuracy, metric\n')
    for i, k in enumerate(results['param_n_neighbors']):
        accuracy = results['mean_test_score'][i]
        metric = results['param_metric'][i]
        txtfile.write(f"{k}, {accuracy}, {metric}\n")
    txtfile.write('\n')
    txtfile.write(f"Best value of k: {best_k}\n")
    txtfile.write(f"Best accuracy score: {best_score}\n")
    txtfile.write(f"Best metric: {best_metric}\n")

print(f"Kết quả đã được lưu vào file {output_file}.")

print(f"⏱ Thời gian chạy GridSearchCV: {elapsed_time:.2f} giây ({minutes} phút {seconds} giây)")
#kết quả là khoảng 2 phút


k = 1, Accuracy = 0.7698, metric = euclidean
k = 1, Accuracy = 0.7698, metric = euclidean
k = 2, Accuracy = 0.7864, metric = euclidean
k = 2, Accuracy = 0.7698, metric = euclidean
k = 3, Accuracy = 0.8086, metric = euclidean
k = 3, Accuracy = 0.8084, metric = euclidean
k = 4, Accuracy = 0.8208, metric = euclidean
k = 4, Accuracy = 0.8139999999999998, metric = euclidean
k = 5, Accuracy = 0.8221999999999999, metric = euclidean
k = 5, Accuracy = 0.8220000000000001, metric = euclidean
k = 6, Accuracy = 0.8367999999999999, metric = euclidean
k = 6, Accuracy = 0.8305999999999999, metric = euclidean
k = 7, Accuracy = 0.8321999999999999, metric = euclidean
k = 7, Accuracy = 0.8333999999999999, metric = euclidean
k = 8, Accuracy = 0.8366, metric = euclidean
k = 8, Accuracy = 0.8357999999999999, metric = euclidean
k = 9, Accuracy = 0.8321999999999999, metric = euclidean
k = 9, Accuracy = 0.8343999999999999, metric = euclidean
k = 10, Accuracy = 0.8442000000000001, metric = euclidean
k = 10, Accu

2.GridSearchCV cho SVM

In [10]:
# Phân lớp bằng SVM với Random Search
# Xác định các tham số cho Random Search

param_grid = {'C': [0.01, 0.1, 1, 10, 20], 'kernel':['linear', 'rbf'],'gamma':['scale', 'auto']}

#param_dist = {
#       'C': np.logspace(-3, 3, 10),
#      'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
#     'gamma': ['scale', 'auto'],
#    'degree': range(2, 6),  # Chỉ dùng khi kernel là 'poly'
#   'coef0': [0.0, 0.1, 0.5, 1.0]  # Chỉ dùng khi kernel là 'poly' hoặc 'sigmoid'
#}


# Tạo mô hình SVM
svm = SVC()

#có 2 hàm GridSearchCV và RandomizedSearchCV để tìm tham số tốt nhất
# GridSearchCV thì ta điền 1 tham số cụ thể ví dụ 'n_neighbors': [3,5,7]
# còn RandomizedSearchCV thì tham số  là 1 list, phạm vi nhiều hơn, ví dụ 'n_neighbors': range(1, 100)
#kết quả cũng như nhau

# Chia dữ liệu thành 10 fold để huấn luyện
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

#random_search = RandomizedSearchCV(svm, param_distributions=param_dist, cv=skf, scoring='accuracy')
#random_search.fit(X, y)

start_time = time.time()


grid_search = GridSearchCV(svm, param_grid, cv=skf, scoring='accuracy')
grid_search.fit(X, y)

end_time = time.time()
elapsed_time = end_time - start_time


# Lấy toàn bộ dữ liệu
results = grid_search.cv_results_

best_svm = grid_search.best_estimator_

print(best_svm)

# In kết quả
for i, C in enumerate(results['param_C']):
    print(f"C = {C}, Accuracy = {results['mean_test_score'][i]}, kernel = {results['param_kernel'][i]}")

# In kết quả tốt nhất
best_C = grid_search.best_params_['C']
best_score = grid_search.best_score_
best_kernel = grid_search.best_params_['kernel']
print(f"\nBest value of C: {best_C}")
print(f"Best accuracy score: {best_score}")
print(f"Best kernel: {best_kernel}")



# Lưu kết quả vào file CSV với ngày giờ chạy
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_file = "/content/drive/MyDrive/Colab Notebooks/HUONG DAN LUAN VAN 2025-2026/DU DOAN UNG THU PHOI/1.2.ChonThamSoToiUu/lungcancer_grid_search_SVM.txt"

with open(output_file, 'w') as txtfile:
    txtfile.write('C, Accuracy, kernel\n')

    for i, C in enumerate(results['param_C']):
        accuracy = results['mean_test_score'][i]
        kernel = results['param_kernel'][i]

        txtfile.write(f"{C}, {accuracy}, {kernel}\n")

    txtfile.write('\n')
    txtfile.write(f"Best value of C: {best_C}\n")
    txtfile.write(f"Best accuracy score: {best_score}\n")
    txtfile.write(f"Best kernel: {best_kernel}\n")

print(f"Kết quả đã được lưu vào file {output_file}.")

# Chuyển giây sang phút + giây
minutes = int(elapsed_time // 60)
seconds = int(elapsed_time % 60)
print(f"⏱ Thời gian chạy GridSearchCV: {elapsed_time:.2f} giây ({minutes} phút {seconds} giây)")
#kết quả là khoảng 3 phút

SVC(C=20, gamma='auto')
C = 0.01, Accuracy = 0.8784000000000001, kernel = linear
C = 0.01, Accuracy = 0.6072, kernel = rbf
C = 0.01, Accuracy = 0.8784000000000001, kernel = linear
C = 0.01, Accuracy = 0.5978, kernel = rbf
C = 0.1, Accuracy = 0.8808, kernel = linear
C = 0.1, Accuracy = 0.8852, kernel = rbf
C = 0.1, Accuracy = 0.8808, kernel = linear
C = 0.1, Accuracy = 0.8904, kernel = rbf
C = 1.0, Accuracy = 0.8874000000000001, kernel = linear
C = 1.0, Accuracy = 0.8932, kernel = rbf
C = 1.0, Accuracy = 0.8874000000000001, kernel = linear
C = 1.0, Accuracy = 0.8814, kernel = rbf
C = 10.0, Accuracy = 0.8886000000000001, kernel = linear
C = 10.0, Accuracy = 0.8904, kernel = rbf
C = 10.0, Accuracy = 0.8886000000000001, kernel = linear
C = 10.0, Accuracy = 0.8947999999999998, kernel = rbf
C = 20.0, Accuracy = 0.8888000000000001, kernel = linear
C = 20.0, Accuracy = 0.8802, kernel = rbf
C = 20.0, Accuracy = 0.8888000000000001, kernel = linear
C = 20.0, Accuracy = 0.8956, kernel = rbf

Best 

3.GridsearchCV cho DT

In [11]:
# Phân lớp bằng DecisionTree với Random Search
# Xác định các tham số cho Random Search
#param_grid = {
    #    'max_depth': range(1, 100),
   #     'min_samples_split': range(2, 20),
   #     'min_samples_leaf': range(1, 20),
   #     'criterion': ['gini', 'entropy']
  #  }


#param_grid = {'max_depth': [None] + list(range(1, 100)), 'criterion':['gini', 'entropy']}
param_grid = {'max_depth': [None] + list(range(1, 100)), 'criterion':['gini', 'entropy']}
# Tạo mô hình DT classifier
dt = DecisionTreeClassifier(random_state=42)


# Chia dữ liệu thành 10 fold để huấn luyện
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

#có 2 hàm GridSearchCV và RandomizedSearchCV để tìm tham số tốt nhất
# GridSearchCV thì ta điền 1 tham số cụ thể ví dụ 'n_neighbors': [3,5,7]
# còn RandomizedSearchCV thì tham số  là 1 list, phạm vi nhiều hơn, ví dụ 'n_neighbors': range(1, 100)
#kết quả cũng như nhau

#random_search = RandomizedSearchCV(dt, param_distributions=param_dist, cv=skf, scoring='accuracy')
#random_search.fit(X, y)


start_time = time.time()
grid_search = GridSearchCV(dt, param_grid, cv=skf, scoring='accuracy')
grid_search.fit(X, y)

end_time = time.time()
elapsed_time = end_time - start_time

# Lấy toàn bộ dữ liệu
results = grid_search.cv_results_

# In kết quả
for i, max_depth in enumerate(results['param_max_depth']):
    print(f"max_depth = {max_depth}, Accuracy = {results['mean_test_score'][i]}, criterion = {results['param_criterion'][i]}")


best_dt = grid_search.best_estimator_

print(best_dt)

# In kết quả tốt nhất
best_max_depth_dct = grid_search.best_params_['max_depth']
best_score = grid_search.best_score_
best_criterion = grid_search.best_params_['criterion']
print(f"\nBest value of max_depth: {best_max_depth_dct}")
print(f"Best accuracy score: {best_score}")
print(f"Best criterion: {best_criterion}")




# Lưu kết quả vào file CSV với ngày giờ chạy
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

output_file = "/content/drive/MyDrive/Colab Notebooks/HUONG DAN LUAN VAN 2025-2026/DU DOAN UNG THU PHOI/1.2.ChonThamSoToiUu/lungcancer_grid_search_DT.txt"

with open(output_file, 'w') as txtfile:
    txtfile.write('max_depth, Accuracy, criterion\n')

    for i, max_depth in enumerate(results['param_max_depth']):
        accuracy = results['mean_test_score'][i]
        criterion = results['param_criterion'][i]

        txtfile.write(f"{max_depth}, {accuracy}, {criterion}\n")

    txtfile.write('\n')
    txtfile.write(f"Best value of max_depth: {best_max_depth_dct}\n")
    txtfile.write(f"Best accuracy score: {best_score}\n")
    txtfile.write(f"Best criterion: {best_criterion}\n")

print(f"Kết quả đã được lưu vào file {output_file}.")

# Chuyển giây sang phút + giây
minutes = int(elapsed_time // 60)
seconds = int(elapsed_time % 60)
print(f"⏱ Thời gian chạy GridSearchCV: {elapsed_time:.2f} giây ({minutes} phút {seconds} giây)")

#kết quả là khoảng 2 phút


max_depth = None, Accuracy = 0.8337999999999999, criterion = gini
max_depth = 1, Accuracy = 0.683, criterion = gini
max_depth = 2, Accuracy = 0.7728, criterion = gini
max_depth = 3, Accuracy = 0.8446, criterion = gini
max_depth = 4, Accuracy = 0.8552, criterion = gini
max_depth = 5, Accuracy = 0.8638, criterion = gini
max_depth = 6, Accuracy = 0.882, criterion = gini
max_depth = 7, Accuracy = 0.8981999999999999, criterion = gini
max_depth = 8, Accuracy = 0.8962, criterion = gini
max_depth = 9, Accuracy = 0.8854, criterion = gini
max_depth = 10, Accuracy = 0.8772, criterion = gini
max_depth = 11, Accuracy = 0.8694000000000001, criterion = gini
max_depth = 12, Accuracy = 0.8652, criterion = gini
max_depth = 13, Accuracy = 0.86, criterion = gini
max_depth = 14, Accuracy = 0.8512000000000001, criterion = gini
max_depth = 15, Accuracy = 0.8502000000000001, criterion = gini
max_depth = 16, Accuracy = 0.8425999999999998, criterion = gini
max_depth = 17, Accuracy = 0.8408000000000001, criterio

4.GridSearchCV cho Random Forest

In [12]:
# Tìm các tham số tối ứu cho random forest
param_grid = {
    'max_depth': [None] + list(range(1, 20)),
    #'n_estimators':[10,20,30,40,50,60,70,80,90,100,200,300,400,500,600,700,800,900],
    'n_estimators':[10,20,30,40,50,60,70,80,90,100,200,300],
}

# Tạo mô hình RandomForestClassifier
rf = RandomForestClassifier(random_state=42)

# Chia dữ liệu thành 10 fold để huấn luyện
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Tìm kiếm
start_time = time.time()

grid_search = GridSearchCV(rf, param_grid, cv=skf, scoring='accuracy')
grid_search.fit(X, y)

end_time = time.time()
elapsed_time = end_time - start_time

#random_search = RandomizedSearchCV(rf, param_distributions=param_dist, cv=skf, scoring='accuracy')
#random_search.fit(X, y)


# Lấy toàn bộ kết quả
results = grid_search.cv_results_

# In kết quả
for i, max_depth in enumerate(results['param_max_depth']):
    n_estimators = results['param_n_estimators'][i]
    accuracy = results['mean_test_score'][i]
    print(f"max_depth = {max_depth}, n_estimators = {n_estimators}, Accuracy = {accuracy}")

# In kết quả tốt nhất
best_max_depth_rf = grid_search.best_params_['max_depth']
best_n_estimators_rf = grid_search.best_params_['n_estimators']
best_score = grid_search.best_score_
print(f"\nBest value of max_depth: {best_max_depth_rf}")
print(f"Best value of n_estimators: {best_n_estimators_rf}")
print(f"Best accuracy score: {best_score}")

# Lưu kết quả vào file CSV với ngày giờ chạy
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_file = f"/content/drive/MyDrive/Colab Notebooks/HUONG DAN LUAN VAN 2025-2026/DU DOAN UNG THU PHOI/1.2.ChonThamSoToiUu/lungcancer_grid_search_RF.txt"


with open(output_file, 'w') as txtfile:
    txtfile.write('max_depth, n_estimators, accuracy\n')

    for i, max_depth in enumerate(results['param_max_depth']):
        n_estimators = results['param_n_estimators'][i]
        accuracy = results['mean_test_score'][i]

        txtfile.write(f"{max_depth}, {n_estimators}, {accuracy}\n")

    txtfile.write('\n')
    txtfile.write(f"Best value of max_depth: {best_max_depth_rf}\n")
    txtfile.write(f"Best value of n_estimators: {best_n_estimators_rf}\n")
    txtfile.write(f"Best accuracy score: {best_score}\n")

print(f"Kết quả đã được lưu vào file {output_file}.")

# Chuyển giây sang phút + giây
minutes = int(elapsed_time // 60)
seconds = int(elapsed_time % 60)
print(f"⏱ Thời gian chạy GridSearchCV: {elapsed_time:.2f} giây ({minutes} phút {seconds} giây)")

#kết quả là khoảng 26 phút

max_depth = None, n_estimators = 10, Accuracy = 0.8934000000000001
max_depth = None, n_estimators = 20, Accuracy = 0.9057999999999999
max_depth = None, n_estimators = 30, Accuracy = 0.908
max_depth = None, n_estimators = 40, Accuracy = 0.9094000000000001
max_depth = None, n_estimators = 50, Accuracy = 0.9109999999999999
max_depth = None, n_estimators = 60, Accuracy = 0.9102
max_depth = None, n_estimators = 70, Accuracy = 0.9112
max_depth = None, n_estimators = 80, Accuracy = 0.9110000000000001
max_depth = None, n_estimators = 90, Accuracy = 0.9110000000000001
max_depth = None, n_estimators = 100, Accuracy = 0.9116000000000002
max_depth = None, n_estimators = 200, Accuracy = 0.9122
max_depth = None, n_estimators = 300, Accuracy = 0.9114000000000001
max_depth = 1, n_estimators = 10, Accuracy = 0.6532000000000001
max_depth = 1, n_estimators = 20, Accuracy = 0.6652
max_depth = 1, n_estimators = 30, Accuracy = 0.6716
max_depth = 1, n_estimators = 40, Accuracy = 0.6936
max_depth = 1, n_estim

5.GridSearchCV cho BagDT

In [13]:
# Tham số cần tìm kiếm
param_grid = {'n_estimators': range(100, 1000, 100)}

# Tạo mô hình DCT_BG
dt = DecisionTreeClassifier(max_depth=best_max_depth_dct, criterion=best_criterion, random_state=42)
bg = BaggingClassifier(dt, bootstrap=True, random_state=42, n_jobs=8)
# Chia dữ liệu thành 10 fold để huấn luyện
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


start_time = time.time()

grid_search = GridSearchCV(bg, param_grid, cv=skf, scoring='accuracy')
grid_search.fit(X, y)

end_time = time.time()
elapsed_time = end_time - start_time

# Lấy toàn bộ dữ liệu
results = grid_search.cv_results_

# In kết quả
for i, n_estimators in enumerate(results['param_n_estimators']):
    print(f"n_estimators = {n_estimators}, Accuracy = {results['mean_test_score'][i]}")

# In kết quả tốt nhất
best_n_estimators_bg = grid_search.best_params_['n_estimators']
best_score = grid_search.best_score_
print(f"\nBest value of n_estimators: {best_n_estimators_bg}")
print(f"Best accuracy score: {best_score}")

# Lưu kết quả vào file CSV với ngày giờ chạy
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

output_file = "/content/drive/MyDrive/Colab Notebooks/HUONG DAN LUAN VAN 2025-2026/DU DOAN UNG THU PHOI/1.2.ChonThamSoToiUu/lungcancer_grid_search_BagDT.txt"

with open(output_file, 'w') as txtfile:
    txtfile.write('n_estimators, Accuracy\n')

    for i, n_estimators in enumerate(results['param_n_estimators']):
        accuracy = results['mean_test_score'][i]

        txtfile.write(f"{n_estimators}, {accuracy}\n")

    txtfile.write('\n')
    txtfile.write(f"Best value of n_estimators: {best_n_estimators_bg}\n")
    txtfile.write(f"Best accuracy score: {best_score}\n")

print(f"Kết quả đã được lưu vào file {output_file}.")

# Chuyển giây sang phút + giây
minutes = int(elapsed_time // 60)
seconds = int(elapsed_time % 60)
print(f"⏱ Thời gian chạy GridSearchCV: {elapsed_time:.2f} giây ({minutes} phút {seconds} giây)")

#kết quả là khoảng 15 phút

n_estimators = 100, Accuracy = 0.9114000000000001
n_estimators = 200, Accuracy = 0.9118
n_estimators = 300, Accuracy = 0.9124000000000001
n_estimators = 400, Accuracy = 0.9120000000000001
n_estimators = 500, Accuracy = 0.9118
n_estimators = 600, Accuracy = 0.9120000000000001
n_estimators = 700, Accuracy = 0.9118
n_estimators = 800, Accuracy = 0.9118
n_estimators = 900, Accuracy = 0.9118

Best value of n_estimators: 300
Best accuracy score: 0.9124000000000001
Kết quả đã được lưu vào file /content/drive/MyDrive/Colab Notebooks/HUONG DAN LUAN VAN 2025-2026/DU DOAN UNG THU PHOI/1.2.ChonThamSoToiUu/lungcancer_grid_search_BagDT.txt.
⏱ Thời gian chạy GridSearchCV: 950.61 giây (15 phút 50 giây)


6. GridSearch cho AdaBoost

In [14]:
# Tham số cần tìm kiếm
param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],
    'learning_rate': [0.01,0.05,0.1,0.5,1]
}

# Tạo mô hình DCT_Ada
dt = DecisionTreeClassifier(max_depth=best_max_depth_dct, criterion=best_criterion, random_state=42)
ada = AdaBoostClassifier(dt, random_state=42)
# Chia dữ liệu thành 10 fold để huấn luyện
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


start_time = time.time()

grid_search = GridSearchCV(ada, param_grid, cv=skf, scoring='accuracy')
grid_search.fit(X, y)

end_time = time.time()
elapsed_time = end_time - start_time


# Lấy toàn bộ dữ liệu
results = grid_search.cv_results_

# In kết quả
for i, n_estimators in enumerate(results['param_n_estimators']):
    print(f"n_estimators = {n_estimators}, Accuracy = {results['mean_test_score'][i]}, learning_rate = {results['param_learning_rate'][i]}")

# In kết quả tốt nhất
best_n_estimators_ada = grid_search.best_params_['n_estimators']
best_score = grid_search.best_score_
best_learning_rate_ada = grid_search.best_params_['learning_rate']

print(f"\nBest value of n_estimators: {best_n_estimators_ada}")
print(f"Best value of learning_rate: {best_learning_rate_ada}")
print(f"Best accuracy score: {best_score}")

# Lưu kết quả vào file CSV với ngày giờ chạy
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

output_file = "/content/drive/MyDrive/Colab Notebooks/HUONG DAN LUAN VAN 2025-2026/DU DOAN UNG THU PHOI/1.2.ChonThamSoToiUu/lungcancer_grid_search_ADA.txt"


with open(output_file, 'w') as txtfile:
    txtfile.write('n_estimators, Accuracy, learning_rate\n')

    for i, n_estimators in enumerate(results['param_n_estimators']):
        accuracy = results['mean_test_score'][i]
        learning_rate = results['param_learning_rate'][i]

        txtfile.write(f"{n_estimators}, {accuracy}, {learning_rate}\n")

    txtfile.write('\n')
    txtfile.write(f"Best value of n_estimators: {best_n_estimators_ada}\n")
    txtfile.write(f"Best value of learning_rate: {best_learning_rate_ada}\n")
    txtfile.write(f"Best accuracy score: {best_score}\n")

print(f"Kết quả đã được lưu vào file {output_file}.")

# Chuyển giây sang phút + giây
minutes = int(elapsed_time // 60)
seconds = int(elapsed_time % 60)
print(f"⏱ Thời gian chạy GridSearchCV: {elapsed_time:.2f} giây ({minutes} phút {seconds} giây)")

#kết quả là khoảng 46 phút

n_estimators = 100, Accuracy = 0.9066000000000001, learning_rate = 0.01
n_estimators = 200, Accuracy = 0.9074000000000002, learning_rate = 0.01
n_estimators = 300, Accuracy = 0.9064, learning_rate = 0.01
n_estimators = 400, Accuracy = 0.9056, learning_rate = 0.01
n_estimators = 500, Accuracy = 0.9046, learning_rate = 0.01
n_estimators = 100, Accuracy = 0.9046, learning_rate = 0.05
n_estimators = 200, Accuracy = 0.9052000000000001, learning_rate = 0.05
n_estimators = 300, Accuracy = 0.9048, learning_rate = 0.05
n_estimators = 400, Accuracy = 0.9052000000000001, learning_rate = 0.05
n_estimators = 500, Accuracy = 0.9044000000000001, learning_rate = 0.05
n_estimators = 100, Accuracy = 0.9046000000000001, learning_rate = 0.1
n_estimators = 200, Accuracy = 0.9036, learning_rate = 0.1
n_estimators = 300, Accuracy = 0.9044000000000001, learning_rate = 0.1
n_estimators = 400, Accuracy = 0.9054, learning_rate = 0.1
n_estimators = 500, Accuracy = 0.9042, learning_rate = 0.1
n_estimators = 100, A

7. GridSearch cho MLP

In [15]:
# Tham số cần tìm kiếm
param_grid = {
    #'max_iter': [500,1000,3000,5000,10000,20000],
    'max_iter': [500,700,1000],
    'hidden_layer_sizes': [(50,), (100,), (200,), (50, 50), (100, 100), (200, 200)]
}

# Tạo mô hình MLP
mlp = MLPClassifier(random_state=42)

# Chia dữ liệu thành 10 fold để huấn luyện
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Tìm kiếm lưới
start_time = time.time()

grid_search = GridSearchCV(mlp, param_grid, cv=skf, scoring='accuracy')
grid_search.fit(X, y)

end_time = time.time()
elapsed_time = end_time - start_time

# Lấy toàn bộ kết quả
results = grid_search.cv_results_

# In kết quả
for i, max_iter in enumerate(results['param_max_iter']):
    hidden_layer_sizes = results['param_hidden_layer_sizes'][i]
    accuracy = results['mean_test_score'][i]
    print(f"max_iter = {max_iter}, hidden_layer_sizes = {hidden_layer_sizes}, Accuracy = {accuracy}")

# In kết quả tốt nhất
best_max_iter = grid_search.best_params_['max_iter']
best_hidden_layer_sizes = grid_search.best_params_['hidden_layer_sizes']
best_score = grid_search.best_score_
print(f"\nBest value of max_iter: {best_max_iter}")
print(f"Best value of hidden_layer_sizes: {best_hidden_layer_sizes}")
print(f"Best accuracy score: {best_score}")

# Lưu kết quả vào file CSV với ngày giờ chạy
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_file = "/content/drive/MyDrive/Colab Notebooks/HUONG DAN LUAN VAN 2025-2026/DU DOAN UNG THU PHOI/1.2.ChonThamSoToiUu/lungcancer_grid_search_MLP.txt"


with open(output_file, 'w') as txtfile:
    txtfile.write('max_iter, hidden_layer_sizes, accuracy\n')

    for i, max_iter in enumerate(results['param_max_iter']):
        hidden_layer_sizes = results['param_hidden_layer_sizes'][i]
        accuracy = results['mean_test_score'][i]

        txtfile.write(f"{max_iter}, {hidden_layer_sizes}, {accuracy}\n")

    txtfile.write('\n')
    txtfile.write(f"Best value of max_iter: {best_max_iter}\n")
    txtfile.write(f"Best value of hidden_layer_sizes: {best_hidden_layer_sizes}\n")
    txtfile.write(f"Best accuracy score: {best_score}\n")

print(f"Kết quả đã được lưu vào file {output_file}.")
# Chuyển giây sang phút + giây
minutes = int(elapsed_time // 60)
seconds = int(elapsed_time % 60)
print(f"⏱ Thời gian chạy GridSearchCV: {elapsed_time:.2f} giây ({minutes} phút {seconds} giây)")

#kết quả là khoảng 71 phút

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptro

max_iter = 500, hidden_layer_sizes = (50,), Accuracy = 0.8986000000000001
max_iter = 700, hidden_layer_sizes = (50,), Accuracy = 0.8943999999999999
max_iter = 1000, hidden_layer_sizes = (50,), Accuracy = 0.8943999999999999
max_iter = 500, hidden_layer_sizes = (100,), Accuracy = 0.8936
max_iter = 700, hidden_layer_sizes = (100,), Accuracy = 0.8874000000000001
max_iter = 1000, hidden_layer_sizes = (100,), Accuracy = 0.8870000000000001
max_iter = 500, hidden_layer_sizes = (200,), Accuracy = 0.8834
max_iter = 700, hidden_layer_sizes = (200,), Accuracy = 0.8764000000000001
max_iter = 1000, hidden_layer_sizes = (200,), Accuracy = 0.8747999999999999
max_iter = 500, hidden_layer_sizes = (50, 50), Accuracy = 0.8646
max_iter = 700, hidden_layer_sizes = (50, 50), Accuracy = 0.8553999999999998
max_iter = 1000, hidden_layer_sizes = (50, 50), Accuracy = 0.8542
max_iter = 500, hidden_layer_sizes = (100, 100), Accuracy = 0.8583999999999999
max_iter = 700, hidden_layer_sizes = (100, 100), Accuracy = 0.

8.GridSearch cho CatBoosting

In [18]:
# Tham số cần tìm kiếm
# param_grid = {
#     'iterations': range(100, 800),
#     'depth': range(3, 10),
#     'learning_rate': [0.01, 0.05, 0.1, 0.2],
#     'verbose': [0]
# }
#param_grid = {
    # 'iterations': [800, 1000],
    # 'depth': [9, 15],
    # 'learning_rate': [0.05, 0.1],
    # 'verbose': [0]
#}

param_grid = {
     'iterations': [100,200,300,400,500,600,700,800,900,1000],
     #'iterations': [100,200,300,400],
     'depth': [5,6,7,8],
     'learning_rate': [0.05],
     'verbose': [0]
}



# Tạo mô hình catboosting

catboost = CatBoostClassifier(random_state=42)


# Chia dữ liệu thành 10 fold để huấn luyện
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Tìm kiếm

start_time = time.time()

grid_search = GridSearchCV(catboost, param_grid, cv=skf, scoring='accuracy')
grid_search.fit(X, y)

end_time = time.time()
elapsed_time = end_time - start_time


# Lấy toàn bộ kết quả
results = grid_search.cv_results_

# In kết quả
for i, depth in enumerate(results['param_depth']):
    iterations = results['param_iterations'][i]
    learning_rate = results['param_learning_rate'][i]
    accuracy = results['mean_test_score'][i]
    print(f"depth = {depth}, iterations = {iterations}, learning_rate = {learning_rate}, Accuracy = {accuracy}")

# In kết quả tốt nhất
best_depth_catboost = grid_search.best_params_['depth']
best_iterations_catboost = grid_search.best_params_['iterations']
best_learning_rate_catboost = grid_search.best_params_['learning_rate']
best_score = grid_search.best_score_

print(f"\nBest value of depth: {best_depth_catboost }")
print(f"Best value of iterations: {best_iterations_catboost}")
print(f"Best value of learning_rate: {best_learning_rate_catboost}")
print(f"Best accuracy score: {best_score}")

# Lưu kết quả vào file CSV với ngày giờ chạy
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_file = f"/content/drive/MyDrive/Colab Notebooks/HUONG DAN LUAN VAN 2025-2026/DU DOAN UNG THU PHOI/1.2.ChonThamSoToiUu/lungcancer_grid_search_CatBoost.txt"



with open(output_file, 'w') as txtfile:
    txtfile.write('max_depth, n_estimators, accuracy\n')

    for i, depth in enumerate(results['param_depth']):
        iterations = results['param_iterations'][i]
        learning_rate = results['param_learning_rate'][i]
        accuracy = results['mean_test_score'][i]

        txtfile.write(f"{depth}, {iterations}, {learning_rate}, {accuracy}\n")

    txtfile.write('\n')
    txtfile.write(f"Best value of depth: {best_depth_catboost}\n")
    txtfile.write(f"Best value of iterations: {best_iterations_catboost}\n")
    txtfile.write(f"Best value of learning_rate: {best_learning_rate_catboost}\n")
    txtfile.write(f"Best accuracy score: {best_score}\n")

print(f"Kết quả đã được lưu vào file {output_file}.")

# Chuyển giây sang phút + giây
minutes = int(elapsed_time // 60)
seconds = int(elapsed_time % 60)
print(f"⏱ Thời gian chạy GridSearchCV: {elapsed_time:.2f} giây ({minutes} phút {seconds} giây)")

#thời gian chạy 20 phút


depth = 5, iterations = 100, learning_rate = 0.05, Accuracy = 0.909
depth = 5, iterations = 200, learning_rate = 0.05, Accuracy = 0.9114000000000001
depth = 5, iterations = 300, learning_rate = 0.05, Accuracy = 0.9106
depth = 5, iterations = 400, learning_rate = 0.05, Accuracy = 0.9094
depth = 5, iterations = 500, learning_rate = 0.05, Accuracy = 0.908
depth = 5, iterations = 600, learning_rate = 0.05, Accuracy = 0.9072000000000001
depth = 5, iterations = 700, learning_rate = 0.05, Accuracy = 0.906
depth = 5, iterations = 800, learning_rate = 0.05, Accuracy = 0.9052
depth = 5, iterations = 900, learning_rate = 0.05, Accuracy = 0.9052
depth = 5, iterations = 1000, learning_rate = 0.05, Accuracy = 0.9036
depth = 6, iterations = 100, learning_rate = 0.05, Accuracy = 0.9101999999999999
depth = 6, iterations = 200, learning_rate = 0.05, Accuracy = 0.9114000000000001
depth = 6, iterations = 300, learning_rate = 0.05, Accuracy = 0.9108
depth = 6, iterations = 400, learning_rate = 0.05, Accura

9.GridSearch cho XGBoost

In [19]:

# Thiết lập tham số cần tìm kiếm
param_grid = {
    #'n_estimators': [100,200,300,400,500,600,700,800,900],
    'n_estimators': [100,200,300,400, 500, 600],
    'max_depth': [1,2,3,4,5,6,7,8,9],
    'learning_rate': [0.05],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

# Tạo mô hình XGBoost
xgb_model = xgb.XGBClassifier(eval_metric='mlogloss')


# Chia dữ liệu thành 10 fold để huấn luyện
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Thực hiện GridSearchCV
start_time = time.time()

grid_search = GridSearchCV(xgb_model, param_grid, cv=skf, scoring='accuracy', verbose=1)
grid_search.fit(X, y)

end_time = time.time()
elapsed_time = end_time - start_time

# Lấy toàn bộ kết quả
results = grid_search.cv_results_

# In kết quả
for i, depth in enumerate(results['param_max_depth']):
    n_estimators = results['param_n_estimators'][i]
    learning_rate = results['param_learning_rate'][i]
    accuracy = results['mean_test_score'][i]
    print(f"max_depth = {depth}, n_estimators = {n_estimators}, learning_rate = {learning_rate}, Accuracy = {accuracy}")

# In kết quả tốt nhất
best_max_depth_xgb  = grid_search.best_params_['max_depth']
best_n_estimators_xgb  = grid_search.best_params_['n_estimators']
best_learning_rate_xgb  = grid_search.best_params_['learning_rate']
best_score = grid_search.best_score_

print(f"\nBest value of max_depth: {best_max_depth_xgb}")
print(f"Best value of n_estimators: {best_n_estimators_xgb}")
print(f"Best value of learning_rate: {best_learning_rate_xgb}")
print(f"Best accuracy score: {best_score}")

# Lưu kết quả vào file CSV với ngày giờ chạy
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_file = f"/content/drive/MyDrive/Colab Notebooks/HUONG DAN LUAN VAN 2025-2026/DU DOAN UNG THU PHOI/1.2.ChonThamSoToiUu/lungcancer_grid_search_XGBost.txt"


with open(output_file, 'w') as txtfile:
    txtfile.write('max_depth, n_estimators, learning_rate, accuracy\n')

    for i, depth in enumerate(results['param_max_depth']):
        n_estimators = results['param_n_estimators'][i]
        learning_rate = results['param_learning_rate'][i]
        accuracy = results['mean_test_score'][i]

        txtfile.write(f"{depth}, {n_estimators}, {learning_rate}, {accuracy}\n")

    txtfile.write('\n')
    txtfile.write(f"Best value of max_depth: {best_max_depth_xgb}\n")
    txtfile.write(f"Best value of n_estimators: {best_n_estimators_xgb}\n")
    txtfile.write(f"Best value of learning_rate: {best_learning_rate_xgb}\n")
    txtfile.write(f"Best accuracy score: {best_score}\n")

print(f"Kết quả đã được lưu vào file {output_file}.")

# Chuyển giây sang phút + giây
minutes = int(elapsed_time // 60)
seconds = int(elapsed_time % 60)
print(f"⏱ Thời gian chạy GridSearchCV: {elapsed_time:.2f} giây ({minutes} phút {seconds} giây)")

#thời gian 5 phút

Fitting 10 folds for each of 54 candidates, totalling 540 fits
max_depth = 1, n_estimators = 100, learning_rate = 0.05, Accuracy = 0.8474
max_depth = 1, n_estimators = 200, learning_rate = 0.05, Accuracy = 0.8768
max_depth = 1, n_estimators = 300, learning_rate = 0.05, Accuracy = 0.8837999999999999
max_depth = 1, n_estimators = 400, learning_rate = 0.05, Accuracy = 0.8905999999999998
max_depth = 1, n_estimators = 500, learning_rate = 0.05, Accuracy = 0.9011999999999999
max_depth = 1, n_estimators = 600, learning_rate = 0.05, Accuracy = 0.9026
max_depth = 2, n_estimators = 100, learning_rate = 0.05, Accuracy = 0.881
max_depth = 2, n_estimators = 200, learning_rate = 0.05, Accuracy = 0.9
max_depth = 2, n_estimators = 300, learning_rate = 0.05, Accuracy = 0.9048
max_depth = 2, n_estimators = 400, learning_rate = 0.05, Accuracy = 0.9034000000000001
max_depth = 2, n_estimators = 500, learning_rate = 0.05, Accuracy = 0.9008
max_depth = 2, n_estimators = 600, learning_rate = 0.05, Accuracy = 